# Placement Prediction Model
Teaches a model to guess if a student gets **Placed** or **NotPlaced**, and shows the chance (%) of placement.

Run the cells from top to bottom (click a cell, press **Shift + Enter**).

| Part | What happens |
|---|---|
| 1 | Load the 3 files (train, validation, test) |
| 2 | Look at the data and clean it |
| 3 | Statistics and graphs (EDA) |
| 4 | Train the model, check it on validation, then test it once |
| 5 | Save the model and try it on a new student |

## Setup (only needed in Google Colab)

In [ ]:
# Change YOUR-USERNAME to your GitHub username. This copies your repository into Colab.
# If the 'data' folder is already here, nothing is copied.
import os, subprocess
REPO_URL = "https://github.com/YOUR-USERNAME/placement-model.git"
if not os.path.exists("data"):
    subprocess.run(["git", "clone", REPO_URL, "placement-model"], check=True)
    os.chdir("placement-model")
print("Working folder:", os.getcwd())
print("Files in data:", os.listdir("data"))

## Part 1: Load the data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
os.makedirs("figs", exist_ok=True)      # graphs will be saved here
os.makedirs("outputs", exist_ok=True)   # tables and the model will be saved here

train = pd.read_csv("data/train.csv")
val   = pd.read_csv("data/validation.csv")
test  = pd.read_csv("data/test.csv")

print("train     :", train.shape)
print("validation:", val.shape)
print("test      :", test.shape)

## Part 2: Understand and clean the data

In [ ]:
# First rows, column types and a quick summary
display(train.head())
print(train.dtypes)

In [ ]:
# Missing values and repeated rows in each file
for name, d in [("train", train), ("validation", val), ("test", test)]:
    print(name, "-> missing values:", int(d.isna().sum().sum()), "| repeated rows:", int(d.duplicated().sum()))

In [ ]:
# Remove repeated rows from train only (validation and test are kept exactly as they are)
before = len(train)
train = train.drop_duplicates().reset_index(drop=True)
print("Rows removed from train:", before - len(train), "| train now:", train.shape)

In [ ]:
# Outliers: count values outside the usual range (IQR rule) in the number columns
num_cols = ["CGPA", "Internships", "Projects", "Workshops/Certifications",
            "AptitudeTestScore", "SoftSkillsRating", "SSC_Marks", "HSC_Marks"]
rows = []
# Internships, Projects and Workshops only have a few possible values (0, 1, 2, 3), so the IQR rule is not meaningful for them
spread_cols = [c for c in num_cols if train[c].nunique() > 5]
print("Checked for outliers:", spread_cols)
print("Skipped (only a few possible values):", [c for c in num_cols if c not in spread_cols])
for c in spread_cols:
    q1, q3 = train[c].quantile([0.25, 0.75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rows.append([c, int(((train[c] < lo) | (train[c] > hi)).sum()), round(lo, 2), round(hi, 2)])
outl = pd.DataFrame(rows, columns=["column", "outliers", "lower_limit", "upper_limit"])
display(outl)

In [ ]:
# Turn words into numbers so the model can read them
def prepare(d):
    d = d.copy()
    d["ExtracurricularActivities"] = d["ExtracurricularActivities"].map({"Yes": 1, "No": 0})
    d["PlacementTraining"]         = d["PlacementTraining"].map({"Yes": 1, "No": 0})
    d["PlacementStatus"]           = d["PlacementStatus"].map({"Placed": 1, "NotPlaced": 0})
    return d

train = prepare(train); val = prepare(val); test = prepare(test)
print("Any blanks created by mistake?", int(train.isna().sum().sum()) + int(val.isna().sum().sum()) + int(test.isna().sum().sum()))
display(train.head())

## Part 3: Statistics and graphs (EDA)

In [ ]:
# Mean, median, mode, spread, smallest and biggest value
stats = train[num_cols].describe().T[["count", "mean", "50%", "std", "min", "max"]]
stats = stats.rename(columns={"50%": "median"})
stats["mode"] = [train[c].mode()[0] for c in num_cols]
stats = stats.round(2)
stats.to_csv("outputs/table_descriptive_stats.csv")
display(stats)

In [ ]:
# How many Placed and NotPlaced
counts = train["PlacementStatus"].map({1: "Placed", 0: "NotPlaced"}).value_counts()
print(counts, "\n", (counts / counts.sum() * 100).round(1))
counts.plot(kind="bar", color=["#8FA3B8", "#2E6F6B"]); plt.title("Placed vs NotPlaced (train)")
plt.ylabel("Students"); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("figs/fig01_placed_count.png", dpi=150); plt.show()

In [ ]:
# Spread of each number column (histograms)
train[num_cols].hist(bins=15, figsize=(12, 8), color="#2E6F6B", edgecolor="white")
plt.suptitle("Spread of each column"); plt.tight_layout()
plt.savefig("figs/fig02_histograms.png", dpi=150); plt.show()

In [ ]:
# Box plots: do Placed students score higher?
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for a, c in zip(ax.ravel(), num_cols):
    sns.boxplot(data=train, x="PlacementStatus", y=c, ax=a, palette=["#8FA3B8", "#2E6F6B"])
    a.set_xticklabels(["NotPlaced", "Placed"]); a.set_xlabel("")
plt.tight_layout(); plt.savefig("figs/fig03_boxplots.png", dpi=150); plt.show()

In [ ]:
# Placement rate for Yes/No columns and for Internships / Projects
fig, ax = plt.subplots(1, 4, figsize=(15, 4))
for a, c in zip(ax, ["PlacementTraining", "ExtracurricularActivities", "Internships", "Projects"]):
    (train.groupby(c)["PlacementStatus"].mean() * 100).plot(kind="bar", ax=a, color="#2E6F6B")
    a.set_ylabel("% placed"); a.set_title(c); a.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.savefig("figs/fig04_placement_rate.png", dpi=150); plt.show()

In [ ]:
# Scatter plot: CGPA vs aptitude score, coloured by result
plt.figure(figsize=(7, 5))
sns.scatterplot(data=train.sample(1500, random_state=1), x="CGPA", y="AptitudeTestScore",
                hue=train.sample(1500, random_state=1)["PlacementStatus"].map({1: "Placed", 0: "NotPlaced"}),
                palette=["#8FA3B8", "#2E6F6B"], alpha=0.7)
plt.title("CGPA vs Aptitude score"); plt.tight_layout()
plt.savefig("figs/fig05_scatter.png", dpi=150); plt.show()

In [ ]:
# Correlation heatmap: which columns move together (and with placement)
corr = train.corr().round(2)
corr.to_csv("outputs/table_correlation.csv")
plt.figure(figsize=(10, 8)); sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, fmt=".2f")
plt.title("Correlation heatmap"); plt.tight_layout()
plt.savefig("figs/fig06_heatmap.png", dpi=150); plt.show()
print(corr["PlacementStatus"].drop("PlacementStatus").sort_values(ascending=False))

## Part 4: Train, validate and test the model

In [ ]:
target = "PlacementStatus"
features = [c for c in train.columns if c != target]
X_train, y_train = train[features], train[target]
X_val,   y_val   = val[features],   val[target]
X_test,  y_test  = test[features],  test[target]
print("The model will use these columns:", features)

In [ ]:
# Two models. The validation file is the FIRST test, used to choose between them.
models = {
    "Logistic Regression": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}
results = []
for name, m in models.items():
    m.fit(X_train, y_train)
    results.append([name,
                    round(accuracy_score(y_train, m.predict(X_train)), 4),
                    round(accuracy_score(y_val,   m.predict(X_val)), 4)])
res = pd.DataFrame(results, columns=["model", "train_accuracy", "validation_accuracy"])
res.to_csv("outputs/table_model_comparison.csv", index=False)
display(res)
best_name = res.sort_values("validation_accuracy", ascending=False).iloc[0]["model"]
best = models[best_name]
print("Chosen model:", best_name)

In [ ]:
# The SECOND test: test.csv, used only once with the chosen model
pred = best.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, target_names=["NotPlaced", "Placed"]))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["NotPlaced", "Placed"]).plot(cmap="Blues")
plt.title("Test result: right and wrong guesses"); plt.tight_layout()
plt.savefig("figs/fig07_confusion_matrix.png", dpi=150); plt.show()

In [ ]:
# Which columns matter most to the model?
if best_name == "Random Forest":
    imp = pd.Series(best.feature_importances_, index=features)
else:
    imp = pd.Series(np.abs(best.named_steps["model"].coef_[0]), index=features)
imp = (imp / imp.sum() * 100).sort_values(ascending=False).round(1)
imp.to_csv("outputs/table_feature_importance.csv", header=["importance_percent"])
imp.sort_values().plot(kind="barh", color="#2E6F6B"); plt.xlabel("Importance (%)")
plt.title("What matters most for placement"); plt.tight_layout()
plt.savefig("figs/fig08_importance.png", dpi=150); plt.show()
display(imp)

## Part 5: Save the model and try it on a new student

In [ ]:
joblib.dump({"model": best, "features": features, "name": best_name}, "outputs/placement_model.pkl")
print("Saved: outputs/placement_model.pkl")

In [ ]:
# Load the saved model (this is what anyone else would do) and test one new student
saved = joblib.load("outputs/placement_model.pkl")
student = pd.DataFrame([{
    "CGPA": 8.0, "Internships": 1, "Projects": 2, "Workshops/Certifications": 1,
    "AptitudeTestScore": 78, "SoftSkillsRating": 4.2, "ExtracurricularActivities": 1,
    "PlacementTraining": 1, "SSC_Marks": 70, "HSC_Marks": 72}])[saved["features"]]
chance = saved["model"].predict_proba(student)[0][1] * 100
print(f"Chance of placement: {chance:.0f}%")
print("Prediction:", "Placed" if chance >= 50 else "NotPlaced")

## Last step: download all graphs and tables

In [ ]:
# Packs the graphs (figs) and tables/model (outputs) into one zip file and downloads it
import shutil
os.makedirs("results_pack", exist_ok=True)
shutil.copytree("figs", "results_pack/figs", dirs_exist_ok=True)
shutil.copytree("outputs", "results_pack/outputs", dirs_exist_ok=True)
shutil.make_archive("results", "zip", "results_pack")
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("Not in Colab. The file results.zip is in the working folder.")